In [3]:
# Import the SentenceTransformer class from the sentence_transformers library
# This is what converts text into vectors (embeddings)
from sentence_transformers import SentenceTransformer

# Import the cosine_similarity function to measure how similar two vectors are
# Returns a score from 0 (completely different) to 1 (identical meaning)
from sklearn.metrics.pairwise import cosine_similarity

# Use pandas to load and query the books dataset
import pandas as pd

# use numpy to save/load the embeddings to disk
import numpy as np

import os

## Loading data

In [ ]:
# Get a dataset from HuggingFace
# https://huggingface.co/datasets/booksouls/goodreads-book-descriptions

from datasets import load_dataset

ds = load_dataset("booksouls/goodreads-book-descriptions")

In [5]:
df = ds["train"].to_pandas()

print(df.shape)
print(df.head())

(1021106, 2)
                                               title  \
0                                        Good Harbor   
1  The Unschooled Wizard (Sun Wolf and Starhawk, ...   
2                               Best Friends Forever   
3                      The Aeneid for Boys and Girls   
4  All's Fairy in Love and War (Avalon: Web of Ma...   

                                         description  
0  Anita Diamant's international bestseller "The ...  
1  Omnibus book club edition containing the Ladie...  
2  Addie Downs and Valerie Adler were eight when ...  
3  Relates in vigorous prose the tale of Aeneas, ...  
4  To Kara's astonishment, she discovers that a p...  


## Generating embeddings

In [ ]:
# Load the pretrained sentence transformer model
# "all-MiniLM-L6-v2" is a small, fast model that produces 384-dimensional vectors

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Convert every book description in the dataset into a vector
# df["description"] selects just the description column
# show_progress_bar=True prints a progress bar — useful since this can take a while
# The result is a 2D numpy array of shape (num_books, 384)

embeddings = model.encode(df["description"].tolist(), show_progress_bar = True)

In [9]:
# Save the embeddings to disk as a .npy file (numpy's binary format)
# Important as embedding is slow, so you do it ONCE and reload it later

np.save("embeddings.npy", embeddings)


## Reload saved embeddings

In [ ]:
# Load pre-computed embeddings instead of recomputing them every run
embeddings = np.load("embeddings.npy")

In [12]:
# Query
# Define a function that takes a plain English query and returns book recommendations
# top_n=5 means return the 5 most similar books by default

def recommend(query, top_n=5):
  # Convert the user's query string into a vector, same way we did the books
  # Result shape: (1, 384)
  query_vec = model.encode([query])  # encode() expects a list, so we wrap query in brackets → [query]

  # Compare the query vector against every book embedding using cosine similarity
  # Result is a 2D array of shape (1, num_books) — one score per book
  # The [0] at the end flattens it to a 1D array of shape (num_books,)
  scores = cosine_similarity(query_vec, embeddings)[0]

  # argsort() returns the indices that would sort the array from lowest to highest
  # [::-1] reverses it so it goes highest to lowest (most similar first)
  # [:top_n] takes only the first N indices (the top N most similar books)
  top_indices = scores.argsort()[::-1][:top_n]

  # Use the indices to look up the corresponding rows in the DataFrame
  # We only return the title, and description columns — not the raw vectors
  return df.iloc[top_indices][["title", "description"]]

In [18]:
# Call the function with a natural language query and print the results
# This will return the 5 books whose descriptions are semantically closest to this query
pd.set_option("display.max_colwidth", None)  # show full text in every column
recommend("a dark mystery set in Victorian London")

,title,description
816041,The Crimes of Jack The Ripper,A pictorial excursion into the dark underbelly of Victorian London.
674781,Oh! Where are Bloody Mary's earrings?,"A mystery set in the court of Queen Victoria, with flashbacks."
218873,Time Tangle,A ghost story set in England.
47536,The Shadow of William Quest (William Quest Victorian Thriller #1),"London 1853 - Where the grand houses of the wealthy lie a stone's throw from the vilest slums and rookeries of the poor.\nA mysterious stranger carrying a swordstick walks the gaslit alleys and night houses seeking vengeance.\nA man determined to fight for justice against all the wrongs of Victorian society.\nWho is the secretive William Quest?\nFollowing Quest's trail from the teeming streets of London to the lonely coast of Norfolk, Inspector Anders of Scotland Yard is determined to uncover the truth.\nThis exciting Victorian thriller takes the reader into the sinister hinterlands of Victorian London as the hunter becomes the hunted. Then to the wild and lonely countryside of Norfolk for an exciting denouement."
139067,Haunted West End Theatres,Reveals the most haunted theatres in London. This book is intended for those who are interested in the shadowy past of London's West End theatres.


In [28]:
results = recommend("a dark mystery set in Victorian London")

print("I recommend the following books: \n")
for _, row in results.iterrows():
  print(f"--> '{row['title']}', which is a {row['description']}\n")

I recommend the following books: 

--> 'The Crimes of Jack The Ripper', which is a A pictorial excursion into the dark underbelly of Victorian London.

--> 'Oh! Where are Bloody Mary's earrings?', which is a A mystery set in the court of Queen Victoria, with flashbacks.

--> 'Time Tangle', which is a A ghost story set in England.

--> 'The Shadow of William Quest (William Quest Victorian Thriller #1)', which is a London 1853 - Where the grand houses of the wealthy lie a stone's throw from the vilest slums and rookeries of the poor.
A mysterious stranger carrying a swordstick walks the gaslit alleys and night houses seeking vengeance.
A man determined to fight for justice against all the wrongs of Victorian society.
Who is the secretive William Quest?
Following Quest's trail from the teeming streets of London to the lonely coast of Norfolk, Inspector Anders of Scotland Yard is determined to uncover the truth.
This exciting Victorian thriller takes the reader into the sinister hinterland